# Create annotation file

In [ ]:
import pandas as pd

import json

In [ ]:
df = pd.read_csv("../data/input/minimal_pairs_sample_ref.tsv", sep="\t")

In [ ]:
df = df.rename(columns={"NK": "nk", "SK": "sk"})

In [ ]:
# add id
df.insert(0, "id", range(1, len(df) + 1))

In [ ]:
# manually define dialect-specific span mappings

span_annotations = {

    1: [
        {"nk": "창가림", "sk": "커튼"},
        {"nk": "들여다볼수", "sk": "들여다볼 수"},
        {"nk": "있을만큼", "sk": "있을 만큼"},
        {"nk": "남겨놓았소", "sk": "남겨 놓았소"}
    ],

    2: [
        {"nk": "초불", "sk": "촛불"},
        {"nk": "켜둔다는것", "sk": "켜둔다는 것"}
    ],

    3: [{"nk": "웨쳤습니다", "sk": "외쳤습니다"}],

    4: [
        {"nk": "일찌기", "sk": "일찍"},
        {"nk": "생각할만큼", "sk": "생각할 만큼"},
        {"nk": "녀자", "sk": "여자"},
        {"nk": "여러번", "sk": "여러 번"},
        {"nk": "알고보면", "sk": "알고 보면"},
        {"nk": "보내고있었다는것", "sk": "보내고 있었다는 것"}
    ],

    5: [
        {"nk": "요우", "sk": "요 위"},
        {"nk": "하불밖", "sk": "이불 밖"}
    ],

    6: [
        {"nk": "우습강스런데", "sk": "우스꽝스러운 점"},
        {"nk": "삼고있었지만", "sk": "삼고 있었지만"},
        {"nk": "안가서", "sk": "안 가서"},
        {"nk": "고쳐주는데", "sk": "고쳐 주는 데"},
        {"nk": "되였다", "sk": "되었다"}
    ],

    7: [{"nk": "지팽이", "sk": "지팡이"}],

    8: [{"nk": "수양", "sk": "숫양"}],

    9: [{"nk": "수집어하였을가", "sk": "수줍어하였을까"}],

    10: [{"nk": "안해", "sk": "아내"}],

    11: [{"nk": "래일", "sk": "내일"}],

    12: [
        {"nk": "뛰여", "sk": "뛰어"},
        {"nk": "책상밑", "sk": "책상 밑"},
        {"nk": "기여", "sk": "기어"},
        {"nk": "난로옆", "sk": "난로 옆"}
    ],

    13: [
        {"nk": "되여", "sk": "되어"},
        {"nk": "밝았을때", "sk": "밝았을 때"},
        {"nk": "발견되였다", "sk": "발견되었다"},
    ],

    14: [
        {"nk": "녀자", "sk": "여자"},
        {"nk": "원탁곁", "sk": "원탁 곁"},
        {"nk": "사진첩우", "sk": "사진첩 위"},
        {"nk": "들여다보고있었는데", "sk": "들여다보고 있었는데"},
        {"nk": "기다리는듯싶었다", "sk": "기다리는 듯 싶었다"}
    ],

    15: [
        {"nk": "땅우", "sk": "땅 위"},
        {"nk": "사라지고말리라", "sk": "사라지고 말리라"}
    ],

    16: [{"nk": "리해할수", "sk": "이해할 수"}],

    17: [
        {"nk": "이딸리아", "sk": "이탈리아"},
        {"nk": "아름다와요", "sk": "아름다워요"},
    ],

    18: [
        {"nk": "사는것", "sk": "사는 것"},
        {"nk": "기뻐뛰여라", "sk": "기뻐 뛰어라"}
    ],

    19: [
        {"nk": "수자", "sk": "숫자"},
        {"nk": "표시하는것", "sk": "표시하는 것"},
    ],

    20: [{"nk": "인차", "sk": "이내"}]

}

In [ ]:
df["spans_json"] = df["id"].map(lambda i: json.dumps(span_annotations[i], ensure_ascii=False))

In [ ]:
df.to_csv("../data/output/minimal_pairs_sample_annotations.tsv", sep="\t", index=False)

# Dialect-aware chrF Score

Instead of treating each sentence as a single evaluation unit, treat annotated dialect-specific spans (and, separately, dialect-invariant spans) as the evaluation units while keeping the standard corpus-level chrF computation unchanged. This is possible because corpus-level chrF operates on a collection of text segments rather than requiring complete sentences.
1.	Separate sentences into dialect-specific and dialect-invariant parts
2.	Compute chrF score only on the parts of the string that are NK or SK specific 
= dialect-specific score
3.	Compute chrF score on parts of the string that should be identical (not NK or SK specific)
= dialect-invariant score
4.	Compute final chrF score that takes into account both the dialect-specific score & dialect-invariant score


In [9]:
import json
import re
import pandas as pd
from difflib import SequenceMatcher
from sacrebleu.metrics import CHRF

## Computes sentence-level chrF3

In [10]:
chrf = CHRF(word_order=0, beta=3)

In [11]:
def chrf3(hyp: str, ref: str) -> float:
    if not hyp.strip() or not ref.strip(): # empty strings return 0
        return 0.0
    return chrf.sentence_score(hyp, [ref]).score

## Finds where an annotated span appears in a sentence

In [12]:
def find_span_offsets(text: str, span: str):
    """Return all character offsets of span in text."""
    return [(m.start(), m.end()) for m in re.finditer(re.escape(span), text)]
    # re.escape() = to prevent special regex characters from being interpreted

## Removes annotated dialect-specific spans from a sentence

In [13]:
def mask_spans(text: str, spans):
    """Remove all annotated dialect-specific spans from a sentence, leaving only the dialect-invariant part"""
    
    mask = [True] * len(text)

    # loop through each annotated span
    for span in spans:
        # find where this span occurs
        for start, end in find_span_offsets(text, span):
            # marks every character belonging to the dialect-specific span as False
            for i in range(start, end):
                mask[i] = False

    # reconstructs the sentence using only the characters marked True, to extract the dialect-invariant portion
    return "".join(ch for ch, keep in zip(text, mask) if keep).strip()

## Approximate which part of the hypothesis corresponds to the annotated SK reference span

Given an annotated span in the reference sentence, find the corresponding span in the hypothesis sentence, even if the hypothesis differs slightly from the reference. Therefore, use SequenceMatcher to approximate which part of the hypothesis corresponds to the annotated SK reference span.

### 1st Attempt (Failed)

In [ ]:
def map_ref_span_to_hyp(ref: str, hyp: str, ref_start: int, ref_end: int):
    """
    Approximate the hypothesis substring corresponding to a reference span
    using character-level SequenceMatcher alignment.

    Parameters
    ----------
    ref:
        Full reference sentence.
    hyp:
        Full hypothesis sentence.
    ref_start, ref_end:
        Character offsets of the annotated span in the reference.

    Returns
    -------
    str
        The aligned hypothesis substring. Returns an empty string when the
        reference span has no corresponding hypothesis text.
    """
    matcher = SequenceMatcher(None, ref, hyp)
    hyp_positions = []

    # matcher.get_opcodes() : what sequence of edit operations transforms the reference string into the hypothesis string?
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        # overlap between this opcode and the annotated span
        overlap_start = max(i1, ref_start)
        overlap_end = min(i2, ref_end)

        if overlap_start < overlap_end:
            hyp_positions.append((j1, j2))

    if not hyp_positions:
        return ""

    # merges all aligned hypothesis regions into one continuous span. 
    hyp_start = min(x[0] for x in hyp_positions)
    hyp_end = max(x[1] for x in hyp_positions)
    
    return hyp[hyp_start:hyp_end].strip()

Because hyp_dialect_spans is too large, mask_spans() removes too much from the hypothesis, which is why the dialect_invariant_chrf3 becomes 0.0, even when the hypothesis and reference are identical.

The code above correctly detected whether an opcode overlapped the annotated span. But it stored the entire hypothesis side of that opcode, not just the overlapping portion.

Example : 

ref = 내일 일을 자랑하지 말아라 \
hyp = 래일 일을 자랑하지 말아라

annotated reference span: 내일 = ref[0:2] overlaps both

* the replace block '내' <-> '래' at ref[0:1]
* ONLY the first character, '일' of the large equal block '일 일을 자랑하지 말아라' at ref[1:2]
  
in the above code, hyp_positions stores (0, 1) correctly. \
however, it stores (1, 14) instead of (1, 2), thus failing to exclude the dialect-invariant part of the text.

### 2nd Attempt (Solved)

In [14]:
def map_ref_span_to_hyp(ref: str, hyp: str, ref_start: int, ref_end: int) -> str:
    """
    Approximate the hypothesis substring corresponding to a reference span
    using character-level SequenceMatcher alignment.

    Parameters
    ----------
    ref:
        Full reference sentence.
    hyp:
        Full hypothesis sentence.
    ref_start, ref_end:
        Character offsets of the annotated span in the reference.

    Returns
    -------
    str
        The aligned hypothesis substring. Returns an empty string when the
        reference span has no corresponding hypothesis text.
    """
    
    # SequenceMatcher compares the two sentences : How do I transform the reference sentence into the hypothesis sentence?
    matcher = SequenceMatcher(None, ref, hyp, autojunk=False)
    hyp_positions: list[tuple[int, int]] = []

    # matcher.get_opcodes() : lists the sequence of edit operations that transform the reference string into the hypothesis string
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        
        # find overlapping spans between the designated reference block (opcode) and the annotated span
        overlap_start = max(i1, ref_start)
        overlap_end = min(i2, ref_end)

        # prevents non-dialect specific parts from being included in 'hyp_positions'
        if overlap_start >= overlap_end:
            continue
        
        # in an "equal" block, the reference and hypothesis have identical text
        if tag == "equal":
            
            # compute where the annotated span lies within the reference block 
            relative_start = overlap_start - i1
            relative_end = overlap_end - i1

            # extract the hypothesis characters at the same relative position
            hyp_overlap_start = j1 + relative_start
            hyp_overlap_end = j1 + relative_end

            # Example: 
            # ref: "이탈리아 거긴 정말 경치가 아름다워요."
            # hyp: "이탈리아는 정말 경치가 아름다워요."
            # annotated span: ref[15:20] "아름다워요"            
            # ref equal block: ref[7:21] " 정말 경치가 아름다워요."
            # hyp equal block: ref[5:19] " 정말 경치가 아름다워요."
            
            # relative position inside the block: 15 - 7 = 8, 20 - 7 = 13
            # return hyp[5+8 : 5+13] -> "아름다워요"

        # a "replace" block may contain different numbers of characters on the reference and hypothesis sides, so their offsets cannot be mapped one-to-one
        elif tag == "replace":
            
            # compute the length of the reference and hypothesis block
            ref_block_len = i2 - i1
            hyp_block_len = j2 - j1

            if ref_block_len == 0 or hyp_block_len == 0:
                continue
            
            # calculate what proportion of the reference block is covered by the annotated span
            relative_start = (overlap_start - i1) / ref_block_len
            relative_end = (overlap_end - i1) / ref_block_len
            
            # project the same proportional range onto the hypothesis block to approximate the corresponding overlapping text
            hyp_overlap_start = j1 + round(relative_start * hyp_block_len)
            hyp_overlap_end = j1 + round(relative_end * hyp_block_len)

            # when a very small annotated span is inside a much larger replace block, rounding may produce an empty span
            # extend it by one character to ensure a non-empty hypothesis substring
            if hyp_overlap_start == hyp_overlap_end:
                hyp_overlap_end = min(j2, hyp_overlap_start + 1)

            # Example: 
            # relevant replacement blocks having different lengths!
            
            # ref : "처음에는 쥴리앙의 우스꽝스러운 점을 ..."
            # hyp : "처음에는 쥴리앙의 우습강스러운데를 ..."
            # annotated span : ref[10:18] "우스꽝스러운 점"            
            # ref equal block : ref[12:19] "꽝스러운 점을"
            # hyp equal block : ref[14:17] "러운데를"
            
            # the proportion of the reference replace block covered by the annotated span : (12 - 12) / 7 = 0, (18 - 12) / 7 = 0.857
            # the same proportional range projected onto the hypothesis block : 14 + round(0 * 3) = 14, 14 + round(0.857 * 3) = 17 
            # hyp_positions = [(10, 11), (13, 14), (14, 17)] 
            # return hyp [10:17] -> "우습강스런데를" 
        
        elif tag == "delete":
            # since the annotated reference span was deleted by the model, there is no hypothesis substring to extract
            continue

        else:
            # "insert" opcodes have no reference-side width (i1 == i2)
            # the hypothesis span corresponds to no reference characters, making overlap_start == overlap_end
            continue
        
        # ignore empty spans and keep only valid hypothesis character ranges
        if hyp_overlap_start < hyp_overlap_end:
            # all collected segments will later be merged into one continuous hypothesis span
            hyp_positions.append((hyp_overlap_start, hyp_overlap_end))

    if not hyp_positions:
        return ""

    # taking the outer boundary reconstructs the corresponding hypothesis span
    hyp_start = min(x[0] for x in hyp_positions)
    hyp_end = max(x[1] for x in hyp_positions)

    # return the hypothesis substring corresponding to the annotated reference span
    return hyp[hyp_start:hyp_end].strip()

## Evaluate each sentence pair

In [ ]:
def evaluate_one_example(ref: str, hyp: str, spans_json: str):
    """
    Compute:
    1. whole-sentence chrF3
    2. dialect-specific chrF3
    3. dialect-invariant chrF3
    4. combined dialect-aware chrF3
    """
    spans = json.loads(spans_json)

    # load annotated SK dialect-specific spans from JSON
    ref_dialect_spans = [s["sk"] for s in spans]

    """
    # extract corresponding hypothesis spans by alignment
    hyp_dialect_spans = []

    for ref_span in ref_dialect_spans:
        offsets = find_span_offsets(ref, ref_span)

        for ref_start, ref_end in offsets:
            hyp_span = map_ref_span_to_hyp(ref, hyp, ref_start, ref_end)
            if hyp_span:
                hyp_dialect_spans.append(hyp_span)
    """
    
    # collect every occurrence of every annotated span, in sentence order
    occurrences = sorted(
        (start, end, span)
        for span in ref_dialect_spans
        for start, end in find_span_offsets(ref, span)
    )

    # build ref and hyp span texts from the same ordered list
    ref_texts, hyp_texts = [], []
    for start, end, span in occurrences:
        ref_texts.append(span)
        hyp_span = map_ref_span_to_hyp(ref, hyp, start, end)
        if hyp_span:
            hyp_texts.append(hyp_span)
            
    ref_dialect_text = " ".join(ref_texts)
    hyp_dialect_text = " ".join(hyp_texts)

    # dialect_specific chrF3 : evaluates whether the model translated the dialect-specific parts correctly
    dialect_specific_score = chrf3(hyp_dialect_text, ref_dialect_text)

    # remove dialect-specific SK spans from reference and hypothesis
    ref_invariant = mask_spans(ref, ref_texts)
    hyp_invariant = mask_spans(hyp, hyp_texts) 

    # dialect-invariant chrF3 : evaluates whether the model translated the non-dialect-specific parts correctly
    dialect_invariant_score = chrf3(hyp_invariant, ref_invariant)

    # whole-sentence chrF3
    whole_sentence_score = chrf3(hyp, ref)
    
    # dialect-aware chrF3
    # rather than assigning a fixed weight to the two component scores, 
    # weight dialect-specific and dialect-invariant chrF according to the proportion of reference characters assigned to each region
    
    # count unique reference character positions covered by dialect-specific spans
    specific_positions = set()
    for start, end, _ in occurrences:
        specific_positions.update(range(start, end))
    
    # number of dialect-specific reference characters
    dialect_specific_char_count = len(specific_positions)   
    
    # number of dialect-invariant reference characters
    dialect_invariant_char_count = len(ref) - dialect_specific_char_count
    total_char_count = len(ref) # total character count
    
    if total_char_count == 0:
        dialect_specific_weight = 0.0
        dialect_invariant_weight = 0.0
        combined_score = 0.0
    else:
        # derive weights from their proportions in the reference
        dialect_specific_weight = (dialect_specific_char_count / total_char_count) # weight for dialect-specific parts
        dialect_invariant_weight = (dialect_invariant_char_count / total_char_count) # weight for dialect-invariant parts
        combined_score = (dialect_specific_weight * dialect_specific_score + dialect_invariant_weight * dialect_invariant_score) # final chrF score      
        
    return {
        "whole_chrF3": whole_sentence_score,
        "dialect_specific_chrF3": dialect_specific_score,
        "dialect_invariant_chrF3": dialect_invariant_score,
        "combined_dialect_aware_chrF3": combined_score,
        "ref_dialect_spans": ref_dialect_text,
        "hyp_dialect_spans": hyp_dialect_text,
        "ref_invariant": ref_invariant,
        "hyp_invariant": hyp_invariant,
    }

* NK reference : 푸께는 너무 일찌기 소원이 이루어졌다고 생각할만큼 녀자의 사랑을 받고 행복을 느낀 일이 여러번 있었지만 알고보면 그 녀자는 번번이 다른 남자에게도 사랑을 보내고있었다는것이다.

* SK reference : 푸께는 너무 일찍 소원이 이루어졌다고 생각할 만큼 여자의 사랑을 받고 행복을 느낀 일이 여러 번 있었지만, 알고 보면 그 여자는 번번이 다른 남자에게도 사랑을 보내고 있었다는 것이다.

* SK hypothesis : 푸께는 너무 일찍 소원이 이루어졌다고 생각할 만큼 녀자의 사랑을 받고 행복을 느낀 일이 여러 번 있었지만, 알고 보면 그 녀자는 번번이 다른 남자에게도 사랑을 보내고 있었다는 것이다.

**Issue 1 (ref side)**

    ref_dialect_text = " ".join(ref_dialect_spans) 

joins each annotation entry exactly once, in annotation order. Since {"nk": "녀자", "sk": "여자"} is one entry, "여자" appears once in ref_dialect_text even though it occurs twice in the reference sentence.

* ref_dialect_spans : 일찍 생각할 만큼 여자 여러 번 알고 보면 보내고 있었다는 것 (INCORRECT) 
  
    should be ->
일찍 생각할 만큼 여자 여러 번 알고 보면 여자 보내고 있었다는 것

**Issue 2 (hyp side)**

    for ref_span in ref_dialect_spans:
            offsets = find_span_offsets(ref, ref_span)

        for ref_start, ref_end in offsets:
            hyp_span = map_ref_span_to_hyp(ref, hyp, ref_start, ref_end)

Because '여자' occurs twice in the reference, there are two different offsets for '여자'. But loop is done over all possible offsets for each ref_span to find the corresponding words in the hypothesis. Accordingly, both '녀자's get extracted and appended consecutively under the same annotation entry, producing "녀자 녀자 ..." instead of being placed in sentence order.

* hyp_dialect_spans : 일찍 생각할 만큼 녀자 녀자 여러 번 알고 보면 보내고 있었다는 것 (INCORRECT) 

    should be ->
일찍 생각할 만큼 녀자 여러 번 알고 보면 녀자 보내고 있었다는 것


**Solution** 

Collected all occurrences with their character positions and sorted by position before building both ref_dialect_texts and hyp_dialect_texts.

Since "여자" is a single annotation entry that matches twice, the second "여자" in position 68 sits before "여러 번" in position 49 and "알고 보면" in position 60. If the spans are joined in this order, ref_dialect_text would end up having two "여자"s glued together.

(7,  9,  일찍), (21, 27, 생각할 만큼), (28, 30, 여자), (68, 70, 여자), (49, 53, 여러 번), (60, 65, 알고 보면), (89, 99, 보내고 있었다는 것)

Sort in increasing order beforehand.

(7,  9,  일찍), (21, 27, 생각할 만큼), (28, 30, 여자), (49, 53, 여러 번), (60, 65, 알고 보면), (68, 70, 여자),(89, 99, 보내고 있었다는 것)

In [16]:
def evaluate_outputs(ref_path, hyp_path, save_path):
    ref = pd.read_csv(ref_path, sep="\t")
    hyp = pd.read_csv(hyp_path, sep="\t")

    # merge the input sentence pairs (gold standard with annotations) with MT outputs
    df = ref.merge(hyp, on="id", how="left")

    rows = []

    for _, row in df.iterrows():
        result = evaluate_one_example(
            ref=row["sk"],
            hyp=row["hyp"],
            spans_json=row["spans_json"]
        )

        rows.append({
            "id": row["id"],
            "nk": row["nk"],
            "ref_sk": row["sk"],
            "hyp_sk": row["hyp"],
            **result
        })

    result_df = pd.DataFrame(rows)
    result_df.to_csv(save_path, sep="\t", index=False)

    # average sentence-level scores across all examples
    summary = result_df[
        [
            "whole_chrF3",
            "dialect_specific_chrF3",
            "dialect_invariant_chrF3",
            "combined_dialect_aware_chrF3"
        ]
    ].mean()

    print("\n===== Mean Sentence-level Results =====")
    print(summary)

    return result_df, summary

In [17]:
if __name__ == "__main__":
    evaluate_outputs(
        ref_path="../data/output/minimal_pairs_sample_annotations.tsv",
        hyp_path="../data/input/minimal_pairs_sample_hyp.tsv",
        save_path="../data/output/dialect_aware_chrf_results_solution1.tsv"
    )


===== Mean Sentence-level Results =====
whole_chrF3                     79.584544
dialect_specific_chrF3          58.866487
dialect_invariant_chrF3         91.249045
combined_dialect_aware_chrF3    86.009739
dtype: float64


#### to understand how SequenceMatcher works

In [ ]:
matcher1 = SequenceMatcher(None, "내일 일을 자랑하지 말아라", "래일 일을 자랑하지 말아라", autojunk=False)

In [ ]:
for tag, i1, i2, j1, j2 in matcher1.get_opcodes():
    print(tag, i1, i2, j1, j2)

In [ ]:
matcher2 = SequenceMatcher(None, "이탈리아 거긴 정말 경치가 아름다워요.", "이딸리아는 정말 경치가 아름다워요.", autojunk=False)

In [ ]:
for tag, i1, i2, j1, j2 in matcher2.get_opcodes():
    print(tag, i1, i2, j1, j2)

In [ ]:
matcher3 = SequenceMatcher(None, "처음에는 쥴리앙의 우스꽝스러운 점을 적당히 대하며 심심풀이로 삼고 있었지만, 얼마 안 가서 이 젊은이의 그릇된 견해를 조금씩 고쳐 주는 데 더 흥미를 느끼게 되었다.", "처음에는 쥴리앙의 우습강스런데를 적당히 대하며 심심풀이로 삼고 있었지만 얼마 안 가서 이 젊은이의 그릇된 견해를 조금씩 고쳐 주는 데 더 흥미를 느끼게 되었다.", autojunk=False)

In [ ]:
for tag, i1, i2, j1, j2 in matcher3.get_opcodes():
    print(tag, i1, i2, j1, j2)